# 🕷️ BioQuery Lab (iNaturalist Edition)

Full disclosure...this was entirely generated with AI (a combination of Claude and Gemini because I ran out of free Claude use for the day). I hope it will work and is both fun and educational, but may need some significant troubleshooting--I encourage you to use AI as needed...

In this exercise you will build and query a real biological database drawn from **iNaturalist** — a citizen-science platform with nearly 300 million biodiversity observations worldwide.

Your instructor has provided an `bioquery_observations.csv` file exported from iNaturalist, filtered to research-grade Florida arachnid observations. The cell in Part 2 has the path on HiPerGator.

You can explore the iNaturalist export tool yourself at: https://www.inaturalist.org/observations/export

---

## ⏱ Suggested Time Budget

| Phase | Time |
|---|---|
| Part 1 — Build the schema | 10 min |
| Part 2 — Load the CSV | 5 min |
| Part 3 — Query challenges | 20 min |
| Part 4 — Written reflection | 10 min |

---

## Part 1 — Build the Schema

Run this cell to define the database models: `Taxon`, `Observer`, and `Observation`.
This uses modern SQLAlchemy 2.0 type-annotated declarations (`Mapped`, `mapped_column`, `String(length)`), ensuring strict, clear schema types in SQLite and Python.

> **✏️ Quick Check:** Before moving on, add a comment below the imports answering: *Why does `Observation` use a foreign key to `Taxon` rather than storing the species name directly?*

In [ ]:
# YOUR QUICK CHECK ANSWER HERE:

from datetime import date
from typing import List, Optional

from sqlalchemy import create_engine, ForeignKey, String, Float, Date, Integer
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

class Base(DeclarativeBase):
    pass

engine = create_engine("sqlite:///biolab.db", echo=False)


class Taxon(Base):
    """Represents a species or broader taxonomic group from iNaturalist."""
    __tablename__ = "taxa"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    common_name: Mapped[Optional[str]] = mapped_column(String(120), nullable=True)
    scientific_name: Mapped[str] = mapped_column(String(120), unique=True, nullable=False)
    kingdom: Mapped[Optional[str]] = mapped_column(String(50), nullable=True)       # e.g., 'Animalia'
    iconic_taxon: Mapped[Optional[str]] = mapped_column(String(50), nullable=True)  # e.g., 'Arachnida'

    observations: Mapped[List["Observation"]] = relationship("Observation", back_populates="taxon")


class Observer(Base):
    """Represents an iNaturalist user who submitted an observation."""
    __tablename__ = "observers"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    username: Mapped[str] = mapped_column(String(100), unique=True, nullable=False)

    observations: Mapped[List["Observation"]] = relationship("Observation", back_populates="observer")


class Observation(Base):
    """A single research-grade field observation from iNaturalist."""
    __tablename__ = "observations"

    # Uses the official iNaturalist integer observation ID as the primary key
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    observed_on: Mapped[Optional[date]] = mapped_column(Date, nullable=True)
    place_guess: Mapped[Optional[str]] = mapped_column(String(255), nullable=True)
    latitude: Mapped[Optional[float]] = mapped_column(Float, nullable=True)
    longitude: Mapped[Optional[float]] = mapped_column(Float, nullable=True)
    quality_grade: Mapped[Optional[str]] = mapped_column(String(30), nullable=True)  # 'research', 'needs_id', 'casual'
    
    taxon_id: Mapped[int] = mapped_column(ForeignKey("taxa.id"), nullable=False)
    observer_id: Mapped[int] = mapped_column(ForeignKey("observers.id"), nullable=False)

    taxon: Mapped["Taxon"] = relationship("Taxon", back_populates="observations")
    observer: Mapped["Observer"] = relationship("Observer", back_populates="observations")


Base.metadata.create_all(engine)
print("Schema created successfully with concrete SQLAlchemy 2.0 types.")

---
## Part 2 — Load the iNaturalist CSV with Data Cleaning

This cell reads your `observations.csv` and populates the database using standard data-cleaning checks:
1. **Required Fields:** Drops records lacking a scientific name.
2. **Type Parsing & Validation:** Safely parses dates into real Python `datetime.date` objects and restricts coordinates to Florida's geographic bounding box (Lat 24.0–31.5, Lon -87.7– -79.9).
3. **Deduplication:** Uses `seen_ids` and internal in-memory caches to prevent duplicate entity insertion.
4. **Reporting:** Counts loaded and skipped rows.

Run this cell once. If you re-run it, delete `biolab.db` first.

In [ ]:
import csv
from datetime import date
from typing import Optional

def parse_coordinate(value: Optional[str], min_val: float, max_val: float) -> Optional[float]:
    if not value:
        return None
    try:
        f = float(value.strip())
        return f if min_val <= f <= max_val else None
    except (ValueError, TypeError):
        return None

def load_inaturalist_csv(filepath: str):
    loaded = 0
    skipped = 0
    seen_ids = set()

    with Session(engine) as session:
        taxon_cache: dict[str, Taxon] = {}      # scientific_name -> Taxon
        observer_cache: dict[str, Observer] = {} # username -> Observer

        with open(filepath, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                # 1. Clean & validate ID
                raw_id = row.get("id", "").strip()
                if not raw_id or not raw_id.isdigit():
                    skipped += 1
                    continue
                obs_id = int(raw_id)
                if obs_id in seen_ids:
                    skipped += 1
                    continue

                # 2. Clean & validate Taxon
                sci_name = row.get("scientific_name", "").strip()
                if not sci_name:
                    skipped += 1
                    continue

                if sci_name not in taxon_cache:
                    c_name = row.get("common_name", "").strip()
                    taxon = Taxon(
                        common_name=c_name if c_name else None,
                        scientific_name=sci_name,
                        kingdom=row.get("taxon_kingdom_name", "").strip() or None,
                        iconic_taxon=row.get("iconic_taxon_name", "").strip() or None,
                    )
                    session.add(taxon)
                    taxon_cache[sci_name] = taxon

                # 3. Clean & validate Observer
                username = row.get("user_login", "").strip() or "unknown_observer"
                if username not in observer_cache:
                    observer = Observer(username=username)
                    session.add(observer)
                    observer_cache[username] = observer

                # 4. Clean Date & Coordinates
                obs_date: Optional[date] = None
                raw_date = row.get("observed_on", "").strip()
                if raw_date:
                    try:
                        obs_date = date.fromisoformat(raw_date)
                    except ValueError:
                        pass

                lat = parse_coordinate(row.get("latitude"), 24.0, 31.5)
                lon = parse_coordinate(row.get("longitude"), -87.7, -79.9)

                obs = Observation(
                    id=obs_id,
                    observed_on=obs_date,
                    place_guess=row.get("place_guess", "").strip() or None,
                    latitude=lat,
                    longitude=lon,
                    quality_grade=row.get("quality_grade", "").strip() or None,
                    taxon=taxon_cache[sci_name],
                    observer=observer_cache[username],
                )
                session.add(obs)
                seen_ids.add(obs_id)
                loaded += 1

        session.commit()
        print(f"Data loaded successfully: {loaded} observations loaded, {skipped} rows skipped.")

load_inaturalist_csv("/blue/bsc4452/share/Class_Files/data/iNaturalist_observations/observations-804395.csv")

---
## Part 3 — Query Challenges

Complete each challenge using SQLAlchemy 2.0 `select()` syntax. After running each query, inspect the results carefully.

---

### 🟢 Challenge 1 — Filter & Join

Retrieve all observations of spiders or scorpions by filtering where `Taxon.common_name` contains `"spider"` or `"scorpion"` (case-insensitively using `.ilike()`). Display the common name, scientific name, and location.

In [ ]:
from sqlalchemy import select, or_

with Session(engine) as session:
    stmt = (
        select(Taxon.common_name, Taxon.scientific_name, Observation.place_guess)
        .join(Observation.taxon)
        .where(or_(
            # YOUR CODE HERE: e.g. Taxon.common_name.ilike("%spider%"), ...
        ))
    )
    results = session.execute(stmt).all()

for common, scientific, place in results:
    print(f"{common or 'No common name'} ({scientific}) — {place or 'Unknown location'}")

---
### 🟡 Challenge 2 — Aggregation

Count the **total number of observations** and the **number of unique species** (`Taxon.scientific_name`).
Group by `Taxon.iconic_taxon` (or by `Observation.place_guess` if all records share the same iconic taxon) and sort by observation count descending.

In [ ]:
from sqlalchemy import select, func

with Session(engine) as session:
    stmt = (
        select(
            Taxon.iconic_taxon,
            # YOUR CODE HERE: func.count(Observation.id) for total observations
            # YOUR CODE HERE: func.count(func.distinct(Taxon.scientific_name)) for unique species
        )
        .join(Observation.taxon)
        .group_by(Taxon.iconic_taxon)
        .order_by(# YOUR CODE HERE: order descending by count
        )
    )
    results = session.execute(stmt).all()

for group, total_obs, unique_spp in results:
    print(f"{group}: {total_obs} observations, {unique_spp} species")

---
### 🔴 Challenge 3 — Open Inquiry

Write your own query that answers an original biological question about the arachnid dataset. Your query must include at least one join and one filter or aggregation.

Possible questions:
- *Which observer has logged the highest number of distinct arachnid species?*
- *Which species has been recorded across the largest number of distinct Florida locations?*
- *What is the seasonal distribution of observations by month?*

In [ ]:
from sqlalchemy import select, func

# YOUR QUESTION (as a comment):

# YOUR QUERY HERE:


---
## Part 4 — Written Reflection

Answer each question in the cell below as Python comments (2–4 sentences each).

In [ ]:
# REFLECTION

# 1. EVIDENCE
#    What pattern did you notice in your Challenge 2 results?
#    Were any arachnid groups or locations noticeably dominant, and what data confirms this?

# 2. INFERENCE
#    Arachnids are often feared or overlooked by the general public.
#    How might this affect who submits arachnid observations on iNaturalist,
#    and what bias could that introduce into the dataset?

# 3. OPEN INQUIRY
#    State your Challenge 3 question and summarize what your results showed.

# 4. LIMITATION
#    iNaturalist data is citizen-science sourced. Name one concrete way this
#    could compromise or limit the scientific conclusions drawn from your queries.